# Работа с transformers

В этом задании мы:
- поработаем с разными моделями huggingface;
- дообучим модель под свою задачу;

# Работа с Huggingface

In [1]:
import os
import random

import numpy as np
import torch


def enable_determinism():
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True)


def fix_seeds(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.mps.manual_seed(seed)


enable_determinism()

In [2]:
import transformers
from transformers import AutoTokenizer


def generate(prompt: str):
    model = "openai-community/gpt2"
    fix_seeds(0)

    ...
    return "some text"


generate("How to make a powerful Artificial Intelligence?")

In [3]:
from transformers import AutoModelForSeq2SeqLM


def translate(source: str):
    model = "Helsinki-NLP/opus-mt-ru-en"
    tokenizer = ...
    model = ...

    input_ids = tokenizer(
        f"переведи с русского на английский: {source}", return_tensors="pt"
    ).input_ids
    outputs = ...
    return ...


translate(
    "Несмотря на то, что в Москве Ростовы принадлежали к высшему обществу, сами того не зная и не думая о том, к какому они принадлежали обществу, в Петербурге общество их было смешанное и неопределенное."
)

## Дообучение моделей

В этой части мы дообучим GPT модель под свои нужды.

Модели для генерации текста обычно устроены следующим образом:
- куча слоев с трансформерами и не только;
- один линейный слой в конце;

На это можно смотреть так: куча слоев выделяют фичи из текста, а линейный слой по ним что-то предсказывает.

Мы можем заменить линейный слой на свой и дообучить под свою задачу.

In [4]:
from transformers import AutoModelForCausalLM
from datasets import load_dataset

# Возьмем IMDB датасет с метками "понравился фильм", "не понравился фильм"
dataset = load_dataset("stanfordnlp/imdb")
train_data = dataset["train"]
test_data = dataset["test"]
# datasets - библиотека для скачивания популярных датасетов.
# Примеры использования:
# print(dataset["text"][12500])
# print(dataset["label"][12500])

# model_name = "google-bert/bert-base-uncased"
# model = AutoModelForCausalLM.from_pretrained(model_name)
# tokenizer = AutoTokenizer.from_pretrained(model_name)

In [6]:
import torch.nn as nn


def create_patched_model():
    model_name = "google-bert/bert-base-uncased"
    ...
    return tokenizer, model

In [7]:
# См. https://huggingface.co/docs/datasets/v1.17.0/use_dataset.html для примеров
def make_loader(split: str = "train"):
    data = dataset[split].map(
        lambda e: tokenizer(e["text"], truncation=True, padding="max_length"),
        batched=True,
    )
    data.set_format(
        type="torch", columns=["input_ids", "token_type_ids", "attention_mask", "label"]
    )
    loader = torch.utils.data.DataLoader(data, batch_size=32, shuffle=True)
    return loader


fix_seeds(0)
train_loader = make_loader("train")
test_loader = make_loader("test")

In [8]:
import torch.nn.functional as F
import tqdm
from torch.optim import Adam


def train_loop(model: nn.Module, loader, device: str = "cpu"):
    # Возьмите тот слой, что вы ответили в прошлом задании
    optimizer = Adam(..., lr=1e-3)
    model.train()
    model.to(device)
    for _ in range(1):
        for one_batch in tqdm.tqdm(loader):
            optimizer.zero_grad()
            target = one_batch.pop("label").to(device)
            pred = model(**{k: v.to(device) for k, v in one_batch.items()})
            pred = pred.logits[:, -1, 0]
            loss = F.binary_cross_entropy_with_logits(pred, target.float())
            loss.backward()
            optimizer.step()


fix_seeds(0)
# Лучше запускать на устройстве с GPU (например, Google Colab), будет намного быстрее
# Поправьте на device="cpu", если запускаете ячейку без GPU
train_loop(model, train_loader, device="cuda")

In [9]:
def save_decoder_layer(model: nn.Module):
    """Сохранить веса последнего линейного слоя в файл model.pt

    Этот файл нужно сдать в соответствующем задании.
    """
    torch.save(model.cls.predictions.decoder.state_dict(), "model.pt")


save_decoder_layer(model)

In [12]:
def accuracy(model, test_loader, device="cpu"):
    model.to(device)
    correct, total = 0, 0
    for one_batch in tqdm.tqdm(test_loader):
        target = one_batch.pop("label").to(device)
        with torch.no_grad():
            predicted = ...
        x = (predicted >= 0.5).int() == target
        correct += x.sum().item()
        total += len(x)
    return correct / total


# Лучше запускать на устройстве с GPU (например, Google Colab), будет намного быстрее
# Поправьте на device="cpu", если запускаете ячейку без GPU
print("accuracy:", accuracy(model, test_loader, device="cuda"))